# Geotecnia II — 01: Bearing capacity of a shallow footing

Starter notebook. It:
1. computes the ultimate bearing capacity of a square footing by hand (Terzaghi, 1943),
2. cross-checks the result with the `geofound` library (Terzaghi and Vesic),
3. plots $q_u$ against footing width,
4. exports the footing plan to a DXF file with `ezdxf` (opens in AutoCAD / LibreCAD).

Change the inputs in the next cell and re-run everything (*Run → Run All Cells*).

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geofound as gf
import ezdxf

OUT = Path("../outputs")
OUT.mkdir(exist_ok=True)

## 1. Inputs (SI: kN, m, kPa, degrees)

In [ ]:
# Soil
phi_deg = 30.0      # friction angle φ' [°]
c       = 10.0      # cohesion c' [kPa]
gamma   = 18.0      # unit weight γ [kN/m³]

# Footing (square)
B  = 2.0            # width [m]
Df = 1.5            # embedment depth [m]
FS = 3.0            # factor of safety

## 2. Terzaghi (1943) by hand — square footing

$$q_u = 1.3\,c\,N_c + q\,N_q + 0.4\,\gamma\,B\,N_\gamma, \qquad q = \gamma D_f$$

with
$N_q = \dfrac{a_\theta^2}{2\cos^2(45° + \phi/2)}$, $a_\theta = e^{(0.75\pi - \phi/2)\tan\phi}$,
$N_c = (N_q - 1)\cot\phi$, and $N_\gamma \approx \dfrac{2(N_q+1)\tan\phi}{1 + 0.4\sin 4\phi}$ (Coduto's approximation).

In [ ]:
def terzaghi_factors(phi_deg):
    phi = math.radians(phi_deg)
    a = math.exp((0.75 * math.pi - phi / 2) * math.tan(phi))
    Nq = a**2 / (2 * math.cos(math.radians(45) + phi / 2) ** 2)
    Nc = (Nq - 1) / math.tan(phi) if phi > 0 else 5.7
    Ng = 2 * (Nq + 1) * math.tan(phi) / (1 + 0.4 * math.sin(4 * phi))
    return Nc, Nq, Ng

def qu_terzaghi_square(c, gamma, B, Df, phi_deg):
    Nc, Nq, Ng = terzaghi_factors(phi_deg)
    q = gamma * Df
    return 1.3 * c * Nc + q * Nq + 0.4 * gamma * B * Ng

Nc, Nq, Ng = terzaghi_factors(phi_deg)
qu = qu_terzaghi_square(c, gamma, B, Df, phi_deg)
q_adm = qu / FS
P_adm = q_adm * B * B

print(f"Nc = {Nc:.2f}   Nq = {Nq:.2f}   Nγ = {Ng:.2f}")
print(f"q_u   = {qu:8.1f} kPa")
print(f"q_adm = {q_adm:8.1f} kPa  (FS = {FS})")
print(f"P_adm = {P_adm:8.1f} kN   (gross, on {B} m × {B} m)")

## 3. Cross-check with `geofound` (works in N, Pa)

In [ ]:
sl = gf.create_soil(phi=phi_deg, cohesion=c * 1e3, unit_dry_weight=gamma * 1e3)
fd = gf.create_foundation(length=B, width=B, depth=Df)

check = pd.DataFrame({
    "q_u [kPa]": {
        "Hand calc (Terzaghi)": qu,
        "geofound Terzaghi 1943": gf.capacity_terzaghi_1943(sl, fd) / 1e3,
        "geofound Vesic 1975": gf.capacity_vesic_1975(sl, fd) / 1e3,
        "geofound Meyerhof 1963": gf.capacity_meyerhof_1963(sl, fd) / 1e3,
    }
}).round(1)
check

## 4. $q_u$ and $q_{adm}$ vs footing width

In [ ]:
widths = np.linspace(0.5, 4.0, 36)
qus = [qu_terzaghi_square(c, gamma, b, Df, phi_deg) for b in widths]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(widths, qus, label="$q_u$ (Terzaghi)")
ax.plot(widths, np.array(qus) / FS, "--", label=f"$q_{{adm}}$ (FS = {FS})")
ax.axvline(B, color="grey", lw=0.8)
ax.set_xlabel("Footing width B [m]")
ax.set_ylabel("Bearing pressure [kPa]")
ax.set_title(f"Square footing, Df = {Df} m, φ' = {phi_deg}°, c' = {c} kPa")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "qu_vs_B.png", dpi=150)
plt.show()

## 5. Export the footing plan to DXF

In [ ]:
doc = ezdxf.new("R2010", setup=True)
doc.units = ezdxf.units.M
msp = doc.modelspace()
doc.layers.add("FOOTING", color=1)
doc.layers.add("TEXT", color=7)

h = B / 2
msp.add_lwpolyline([(-h, -h), (h, -h), (h, h), (-h, h)], close=True, dxfattribs={"layer": "FOOTING"})
msp.add_circle((0, 0), 0.05, dxfattribs={"layer": "FOOTING"})
msp.add_text(
    f"Zapata {B:.2f} x {B:.2f} m  Df={Df} m  q_adm={q_adm:.0f} kPa",
    height=0.1, dxfattribs={"layer": "TEXT"},
).set_placement((-h, -h - 0.3))

dxf_path = OUT / "zapata.dxf"
doc.saveas(dxf_path)
print("Saved", dxf_path.resolve())